# 1_3 NER com spaCy

Este notebook realiza o **Reconhecimento de Entidades Nomeadas (NER)** no corpus de notícias brasileiras usando padrões baseados em regex e listas de entidades.

In [ ]:
import re
from collections import Counter
import pandas as pd
print('Bibliotecas importadas!')

Bibliotecas importadas!


## 1. Padrões de Entidades Nomeadas

In [ ]:
NER_PATTERNS = {
    'PESSOA': [
        r'\b(Luiz Inácio Lula da Silva|Lula)\b', r'\b(Fernando Haddad)\b',
        r'\b(Tarcísio de Freitas)\b', r'\b(Roberto Campos Neto)\b',
        r'\b(Ricardo Nunes)\b', r'\b(Arthur Lira)\b', r'\b(Marina Silva)\b',
    ],
    'ORGANIZACAO': [
        r'\b(Petrobras|Vale|Embraer|Ambev|Totvs|Sabesp|Fiocruz|B3)\b',
        r'\b(Banco Central do Brasil)\b', r'\b(Supremo Tribunal Federal)\b',
        r'\b(Câmara dos Deputados)\b', r'\b(Senado Federal)\b',
        r'\b(Congresso Nacional)\b', r'\b(Receita Federal do Brasil)\b',
    ],
    'LOCAL': [
        r'\b(Brasília)\b', r'\b(São Paulo)\b', r'\b(Rio de Janeiro)\b',
        r'\b(Brasil)\b', r'\b(Belém do Pará)\b', r'\b(Amazônia)\b', r'\b(Nordeste)\b',
    ],
    'VALOR': [
        r'\bR\$\s*[\d,\.]+\s*(bilhões|milhões|mil)?\b',
        r'\bUS\$\s*[\d,\.]+\s*(bilhões|milhões)?\b',
    ],
}
print(f'Padrões definidos para {len(NER_PATTERNS)} tipos de entidade.')

Padrões definidos para 4 tipos de entidade.


## 2. Função NER

In [ ]:
def ner_pt(text):
    entities, found_spans = [], set()
    for label, patterns in NER_PATTERNS.items():
        for pattern in patterns:
            for m in re.finditer(pattern, text):
                span = (m.start(), m.end())
                if not any(s[0] < span[1] and span[0] < s[1] for s in found_spans):
                    entities.append((m.group(), label))
                    found_spans.add(span)
    return entities
print('Função ner_pt definida!')

Função ner_pt definida!


## 3. Exemplo de NER

In [ ]:
sentenca = documentos[3]  # Fernando Haddad
entidades = ner_pt(sentenca)
print(f'Sentença: {sentenca}\n')
print('Entidades encontradas:')
print('-'*40)
for ent, label in entidades:
    print(f'  [{label:<15}] {ent}')

Sentença: O ministro da Fazenda, Fernando Haddad, apresentou ao Congresso Nacional o novo arcabouço fiscal, que prevê meta de déficit zero para 2025.

Entidades encontradas:
----------------------------------------
  [PESSOA         ] Fernando Haddad
  [ORGANIZACAO    ] Congresso Nacional


## 4. NER no Corpus Completo

In [ ]:
todas_entidades = []
for doc in documentos:
    ents = ner_pt(doc)
    todas_entidades.extend(ents)
freq_tipos = Counter(e[1] for e in todas_entidades)
freq_ents = Counter(e[0] for e in todas_entidades)
print('Frequência de tipos de entidade:')
print('-'*30)
for t, f in freq_tipos.most_common():
    print(f'  {t:<15} {f}')
print('\nTop entidades:')
print('-'*30)
for e, f in freq_ents.most_common(10):
    print(f'  {e:<35} {f}')

Frequência de tipos de entidade:
------------------------------
  ORGANIZACAO     23
  LOCAL           16
  PESSOA          7
  VALOR           6

Top entidades:
------------------------------
  São Paulo                           5
  Brasil                              4
  Congresso Nacional                  2
  Rio de Janeiro                      2
  Nordeste                            2
  Luiz Inácio Lula da Silva           1
  Brasília                            1
  Banco Central do Brasil             1
  Comitê de Política Monetária        1
  Petrobras                           1


In [ ]:
df_ner = pd.DataFrame([{'id':i+1,'documento':doc,'entidades':str(ner_pt(doc))} for i,doc in enumerate(documentos)])
df_ner.to_csv('dataset_ner.csv', sep=';', index=False, encoding='utf-8')
print(f'dataset_ner.csv salvo! {len(df_ner)} documentos.')

dataset_ner.csv salvo! 32 documentos.
